# Backtesting Example

Demonstrates `run_backtest()` for single and batch VaR backtesting with parallel execution.

In [1]:
import numpy as np
from risk_backtest import run_backtest, BacktestConfig, BacktestResult

## Generate Synthetic Data

In [2]:
rng = np.random.default_rng(42)
n = 600  # ~2.5 years daily data

# Simulate returns with occasional VaR breaches
returns = rng.normal(0, 0.01, n)
var_series = np.full(n, 0.02)  # Constant 2% daily VaR

# Inject a cluster of breaches
returns[100:105] = -0.03
returns[300] = -0.04
returns[400] = -0.025

print(f"Returns: {n} observations, std={returns.std():.4f}")
print(f"VaR breaches: {(returns < -var_series).sum()}")

Returns: 600 observations, std=0.0102
VaR breaches: 20


## Single Fund Backtest

In [3]:
# Default config: 99% daily VaR
result = run_backtest(
    returns=returns,
    risk_series=var_series,
    window_sizes=[252, 504],  # 1-year and 2-year windows
    cluster_threshold=5,
)

print(result)
result.summary

BacktestResult(series=1, rows=4, config=VaR@0.99)


N_Clusters  N_Isolated  N_Obs  \
name    window_size cluster_adj                                       
default 252         Cluster_Adjusted           0           8    252   
                    Unadjusted                 0           8    252   
        504         Cluster_Adjusted           0          15    504   
                    Unadjusted                 2          13    504   

                                      N_Overshoots  Binomial_Probability  \
name    window_size cluster_adj                                            
default 252         Cluster_Adjusted             8          3.104062e-03   
                    Unadjusted                   8          3.104062e-03   
        504         Cluster_Adjusted            15          1.563904e-04   
                    Unadjusted                  20          2.421279e-07   

                                      Binomial_Cumulative_Prob  \
name    window_size cluster_adj                                  
default 252         Cluster_Adjusted              4.220641e-03   
                    Unadjusted                    4.220641e-03   
        504         Cluster_Adjusted              2.237977e-04   
                    Unadjusted                    3.143060e-07   

                                      Z_Test_Statistic      Z_Test_P  \
name    window_size cluster_adj                                        
default 252         Cluster_Adjusted          3.469466  2.607466e-04   
                    Unadjusted                3.469466  2.607466e-04   
        504         Cluster_Adjusted          4.458887  4.119316e-06   
                    Unadjusted                6.697284  1.061640e-11   

                                      Kupiec_Statistic      Kupiec_P  \
name    window_size cluster_adj                                        
default 252         Cluster_Adjusted          7.644185  5.695561e-03   
                    Unadjusted                7.644185  5.695561e-03   
        504         Cluster_Adjusted         12.999477  3.115781e-04   
                    Unadjusted               25.666135  4.058876e-07   

                                      Christoffersen_Statistic  \
name    window_size cluster_adj                                  
default 252         Cluster_Adjusted                  0.000000   
                    Unadjusted                        0.000000   
        504         Cluster_Adjusted                  0.000000   
                    Unadjusted                        7.684275   

                                      Christoffersen_P  Joint_Test_Statistic  \
name    window_size cluster_adj                                                
default 252         Cluster_Adjusted           1.00000              7.644185   
                    Unadjusted                 1.00000              7.644185   
        504         Cluster_Adjusted           1.00000             12.999477   
                    Unadjusted                 0.00557             33.350410   

                                      Joint_Test_P  Martingale_Test_Statistic  \
name    window_size cluster_adj                                                 
default 252         Cluster_Adjusted  2.188196e-02                   1.416572   
                    Unadjusted        2.188196e-02                   1.416572   
        504         Cluster_Adjusted  1.503833e-03                   2.390957   
                    Unadjusted        5.728626e-08                  22.937085   

                                      Martingale_Test_P  
name    window_size cluster_adj                          
default 252         Cluster_Adjusted           0.922492  
                    Unadjusted                 0.922492  
        504         Cluster_Adjusted           0.792820  
                    Unadjusted                 0.000347

In [4]:
# Pass rates (% of tests passing at 5% significance)
result.pass_rates

Z_Test_P  Kupiec_P  Christoffersen_P  \
window_size cluster_adj                                              
252         Cluster_Adjusted       0.0       0.0             100.0   
            Unadjusted             0.0       0.0             100.0   
504         Cluster_Adjusted       0.0       0.0             100.0   
            Unadjusted             0.0       0.0               0.0   

                              Joint_Test_P  Martingale_Test_P  
window_size cluster_adj                                        
252         Cluster_Adjusted           0.0              100.0  
            Unadjusted                 0.0              100.0  
504         Cluster_Adjusted           0.0              100.0  
            Unadjusted                 0.0                0.0

## Batch Mode (Multiple Funds in Parallel)

In [5]:
# Create synthetic multi-fund data
n_funds = 20
returns_dict = {}
var_dict = {}

for i in range(n_funds):
    fund_name = f"FUND_{i:03d}"
    vol = rng.uniform(0.005, 0.015)
    returns_dict[fund_name] = rng.normal(0, vol, 600)
    var_dict[fund_name] = np.full(600, vol * 2.33)  # ~99% normal VaR

print(f"Funds: {len(returns_dict)}")

Funds: 20


In [6]:
# Run all funds in parallel
import time

t0 = time.time()
batch_result = run_backtest(
    returns=returns_dict,
    risk_series=var_dict,
    window_sizes=[252, 504],
    cluster_threshold=5,
    n_jobs=-1,  # use all CPU cores
)
print(f"Elapsed: {time.time()-t0:.2f}s")
print(batch_result)
batch_result.summary.head(10)

Elapsed: 5.89s
BacktestResult(series=20, rows=80, config=VaR@0.99)


N_Clusters  N_Isolated  N_Obs  \
name     window_size cluster_adj                                       
FUND_000 252         Cluster_Adjusted           0           2    252   
                     Unadjusted                 0           2    252   
         504         Cluster_Adjusted           0           4    504   
                     Unadjusted                 1           3    504   
FUND_001 252         Cluster_Adjusted           0           4    252   
                     Unadjusted                 0           4    252   
         504         Cluster_Adjusted           0           8    504   
                     Unadjusted                 1           7    504   
FUND_002 252         Cluster_Adjusted           0           6    252   
                     Unadjusted                 0           6    252   

                                       N_Overshoots  Binomial_Probability  \
name     window_size cluster_adj                                            
FUND_000 252         Cluster_Adjusted             2              0.256356   
                     Unadjusted                   2              0.256356   
         504         Cluster_Adjusted             4              0.174552   
                     Unadjusted                   5              0.176316   
FUND_001 252         Cluster_Adjusted             4              0.135685   
                     Unadjusted                   4              0.135685   
         504         Cluster_Adjusted             8              0.066793   
                     Unadjusted                   9              0.037182   
FUND_002 252         Cluster_Adjusted             6              0.028268   
                     Unadjusted                   6              0.028268   

                                       Binomial_Cumulative_Prob  \
name     window_size cluster_adj                                  
FUND_000 252         Cluster_Adjusted                  0.718330   
                     Unadjusted                        0.718330   
         504         Cluster_Adjusted                  0.741960   
                     Unadjusted                        0.567407   
FUND_001 252         Cluster_Adjusted                  0.246187   
                     Unadjusted                        0.246187   
         504         Cluster_Adjusted                  0.136535   
                     Unadjusted                        0.069742   
FUND_002 252         Cluster_Adjusted                  0.042523   
                     Unadjusted                        0.042523   

                                       Z_Test_Statistic  Z_Test_P  \
name     window_size cluster_adj                                    
FUND_000 252         Cluster_Adjusted         -0.329219  0.629005   
                     Unadjusted               -0.329219  0.629005   
         504         Cluster_Adjusted         -0.465587  0.679244   
                     Unadjusted               -0.017907  0.507144   
FUND_001 252         Cluster_Adjusted          0.937009  0.174377   
                     Unadjusted                0.937009  0.174377   
         504         Cluster_Adjusted          1.325131  0.092564   
                     Unadjusted                1.772811  0.038130   
FUND_002 252         Cluster_Adjusted          2.203238  0.013789   
                     Unadjusted                2.203238  0.013789   

                                       Kupiec_Statistic  Kupiec_P  \
name     window_size cluster_adj                                    
FUND_000 252         Cluster_Adjusted          0.116636  0.732712   
                     Unadjusted                0.116636  0.732712   
         504         Cluster_Adjusted          0.233272  0.629108   
                     Unadjusted                0.000322  0.985694   
FUND_001 252         Cluster_Adjusted          0.745081  0.388038   
                     Unadjusted                0.745081  0.388038   
         504         Cluster_Adjusted          1.490162  0.2221

## Custom Configuration (Annual Volatility)

In [7]:
# If your risk measure is annual volatility (1-sigma)
config = BacktestConfig(
    risk_measure="volatility",
    confidence_level=0.8413,  # 1-sigma = 84.13%
    horizon="annual",
)

print(f"Config summary: {config.summary()}")
print(f"Scaling factor (annual→daily): {config.scaling_factor:.6f}")
print(f"Daily breach probability: {config.daily_breach_probability:.4f}")

# Backtest with annual vol as risk series
annual_vol = np.full(500, 0.15)  # 15% annual
returns_vol = rng.normal(0, 0.15/np.sqrt(252), 500)

vol_result = run_backtest(returns_vol, annual_vol, config=config, window_sizes=[252])
vol_result.summary

Config summary: {'risk_measure': 'volatility', 'confidence_level': 0.8413, 'horizon': 'annual', 'scaling_method': 'sqrt_t', 'horizon_days': 252, 'scaling_factor': 0.0629940788348712, 'daily_breach_probability': 0.15869999999999995}
Scaling factor (annual→daily): 0.062994
Daily breach probability: 0.1587


N_Clusters  N_Isolated  N_Obs  \
name    window_size cluster_adj                                       
default 252         Cluster_Adjusted           0          18    252   
                    Unadjusted                11           7    252   

                                      N_Overshoots  Binomial_Probability  \
name    window_size cluster_adj                                            
default 252         Cluster_Adjusted            18              0.000016   
                    Unadjusted                  41              0.066936   

                                      Binomial_Cumulative_Prob  \
name    window_size cluster_adj                                  
default 252         Cluster_Adjusted                  0.999990   
                    Unadjusted                        0.457403   

                                      Z_Test_Statistic  Z_Test_P  \
name    window_size cluster_adj                                    
default 252         Cluster_Adjusted         -3.791477  0.999925   
                    Unadjusted                0.173710  0.431047   

                                      Kupiec_Statistic  Kupiec_P  \
name    window_size cluster_adj                                    
default 252         Cluster_Adjusted         17.451692  0.000029   
                    Unadjusted                0.029972  0.862554   

                                      Christoffersen_Statistic  \
name    window_size cluster_adj                                  
default 252         Cluster_Adjusted                  0.000000   
                    Unadjusted                        0.524755   

                                      Christoffersen_P  Joint_Test_Statistic  \
name    window_size cluster_adj                                                
default 252         Cluster_Adjusted           1.00000             17.451692   
                    Unadjusted                 0.46882              0.554727   

                                      Joint_Test_P  Martingale_Test_Statistic  \
name    window_size cluster_adj                                                 
default 252         Cluster_Adjusted      0.000162                   7.431084   
                    Unadjusted            0.757779                   2.689064   

                                      Martingale_Test_P  
name    window_size cluster_adj                          
default 252         Cluster_Adjusted           0.190503  
                    Unadjusted                 0.747790